# 12 — Product modes, SQL analytics, table lifecycle and OCR routing

This acceptance notebook makes the next-level architecture inspectable. It does not claim production OCR quality: the scan section validates routing only.

In [ ]:
from pathlib import Path
import sys
root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table
ROOT = bootstrap()
from app.domain.models import Mode, QuestionRequest, ScopeSelection, EvidenceConstraints
from app.services.engine import engine, retrieval_budget
from ingestion.promote import assess_cell_for_promotion
from ingestion.profiles import detect_document_profile
from ingestion.models import DocumentProfile
print('Project root:', ROOT)

## 1. The three UI modes now change retrieval

Quick limits latency; deep starts at top-1 and expands furthest; summary starts with a wider batch because it is intended for multi-evidence briefs.

In [ ]:
mode_rows = [{'mode': mode.value, 'k_history': list(retrieval_budget(mode))} for mode in Mode]
display_table(mode_rows)
assert retrieval_budget(Mode.QUICK) != retrieval_budget(Mode.DEEP)
assert retrieval_budget(Mode.SUMMARY) != retrieval_budget(Mode.DEEP)

## 2. SQL is used for exhaustive tabular comparison

The planner uses an allow-listed SELECT template with bound parameters. It queries only promoted facts and exposes the SQL trace and source-linked rows.

In [ ]:
request = QuestionRequest(
    question='Compare les ratios de couverture SCR des trois entités en 2025',
    mode='deep',
    scope=ScopeSelection(document_ids=['foyer_group_qrt_2025', 'foyer_assurances_qrt_2025', 'foyer_global_health_qrt_2025']),
    constraints=EvidenceConstraints(period='2025'),
)
answer = await engine.answer(request)
display_table([{'entity': item.fact.entity, 'value': item.fact.formatted_value, 'page': item.source.page, 'row': item.source.row} for item in answer.evidence])
print(answer.summary)
print(answer.analytical_trace.model_dump())
assert answer.retrieval_run.strategy == 'sql_analytics'
assert answer.analytical_trace.row_count == 3

## 3. A cell is not evidence merely because it was extracted

The lifecycle is EXTRACTED → CANDIDATE → VALIDATED → PROMOTED. This prototype combines the last deterministic validations in `assess_cell_for_promotion`; missing provenance or an unexpected column prevents promotion.

In [ ]:
valid = {'row_code':'R0690','column_code':'C0010','normalized_value':2.87,'value_type':'ratio','provenance':{'page':7,'bbox':[1,2,3,4]}}
invalid = {**valid, 'column_code':'C9999', 'provenance':{'page':7}}
promotion_rows = [assess_cell_for_promotion(valid), assess_cell_for_promotion(invalid)]
display_table(promotion_rows)
assert promotion_rows[0]['state'] == 'PROMOTED'
assert promotion_rows[1]['state'] == 'CANDIDATE'

## 4. Scan routing is explicit but OCR accuracy remains to be benchmarked

A document with little native text on at least two of the first three pages is routed to the scanned-document profile. Docling then enables OCR and table structure. This check validates the dispatcher, not character or cell accuracy.

In [ ]:
profile = detect_document_profile(ROOT / 'synthetic-scan.pdf', '', potential_scan_ratio=1.0)
print({'profile': profile.value, 'ocr_expected': profile == DocumentProfile.SCANNED_DOCUMENT})
assert profile == DocumentProfile.SCANNED_DOCUMENT

## 5. Acceptance criterion

The feature is acceptable when mode budgets differ, the SQL comparison returns every in-scope promoted ratio with provenance, unsafe cells are not promoted, and scan-like PDFs are routed to OCR. Production claims still require a representative scanned-PDF benchmark and independently reviewed labels.